In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go

#Data Loading And Cleaning

df = pd.read_csv("dataset.csv")
df_clean = df.drop_duplicates(subset=["track_name","artists"])
df_clean = df.dropna(subset=["artists","album_name","track_name"])

target_genres = ['classical', 'black-metal', 'dance', 'acoustic', 'hip-hop','indian','rock','pop-film']
df_filtered = df_clean[df_clean['track_genre'].isin(target_genres)].copy()

print(f"Dataset shape: {df_filtered.shape}")
print(f"Genres included: {df_filtered['track_genre'].unique()}")

# Feature Extraction And Scaling
audio_features = ["danceability", "energy",  "loudness", "speechiness",
                  "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

X = df_filtered[audio_features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Genre"] = df_filtered['track_genre'].values

print(f"\nPCA variance explained: {pca.explained_variance_ratio_}")
print(pca_df.tail())

Dataset shape: (8000, 21)
Genres included: ['acoustic' 'black-metal' 'classical' 'dance' 'hip-hop' 'indian'
 'pop-film' 'rock']

PCA variance explained: [0.35438297 0.18104928]
           PC1      PC2 Genre
7995  1.365691 -1.22322  rock
7996  1.365691 -1.22322  rock
7997  1.365691 -1.22322  rock
7998  1.365691 -1.22322  rock
7999  1.365691 -1.22322  rock


In [4]:

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="Genre",
    color_discrete_sequence=px.colors.qualitative.Bold,
    labels={'PC1' : "Principal Component 1 ",'PC2' : "Principal Component 2"},
    title="<b> Music Latent Space (2D PCA Projection) </b>",
    opacity=0.5
)

fig.update_traces(marker=dict(size=8))
fig.update_layout(
    template='plotly_dark',
    legend_title_text="Music Genres",
    dragmode="pan",
    plot_bgcolor="rgba(15, 15, 30, 0.5)",
    paper_bgcolor="rgba(15, 15, 30, 0)",
    font=dict(family="Poppins", size=12, color="#ffffff"),
    title_font_size=20,
)

fig.show()

# Audio Features Distribution by Genre
import plotly.graph_objects as go

audio_features = ["danceability", "energy",  "loudness", "speechiness",
                  "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

df_filtered_reset = df_filtered.reset_index(drop=True)

for selected_feature in ["danceability", "energy", "valence"]:
    fig = go.Figure()
    colors_gradient = ['#FF006E', '#FB5607', '#FF8C00', '#FFB703', '#FB5607']
    
    for i, genre in enumerate(sorted(df_filtered_reset['track_genre'].unique())):
        genre_data = df_filtered_reset[df_filtered_reset['track_genre'] == genre][selected_feature]
        fig.add_trace(go.Box(
            y=genre_data, 
            name=genre.upper(),
            marker_color=colors_gradient[i % len(colors_gradient)],
            boxmean='sd'
        ))
    
    fig.update_layout(
        title=f"<b>{selected_feature.capitalize()} Distribution by Genre</b>",
        yaxis_title=selected_feature.capitalize(),
        xaxis_title="Genre",
        template="plotly_dark",
        showlegend=True,
        plot_bgcolor="rgba(15, 15, 30, 0.5)",
        paper_bgcolor="rgba(15, 15, 30, 0)",
        font=dict(family="Poppins", size=12, color="#ffffff"),
        title_font_size=18,
        height=500
    )
    fig.show()

# Genre Distribution (Donut Chart)
genre_counts = df_filtered_reset['track_genre'].value_counts().sort_values(ascending=False)
fig = px.pie(
    values=genre_counts.values,
    names=genre_counts.index,
    title="<b>Genre Distribution in Your Collection</b>",
    color_discrete_sequence=px.colors.qualitative.Bold,
    template="plotly_dark",
    hole=0.3
)
fig.update_layout(
    plot_bgcolor="rgba(15, 15, 30, 0)",
    paper_bgcolor="rgba(15, 15, 30, 0)",
    font=dict(family="Poppins", size=12, color="#ffffff"),
    title_font_size=18,
)
fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    marker=dict(line=dict(color='rgba(15, 15, 30, 0.8)', width=2))
)
fig.show()

similarity_matrix = cosine_similarity(X_pca)
df_filtered = df_filtered.reset_index(drop=True)

def recommend_song_with_filter(song_title, num_recommendation=5):
    try:
        # Find the row index and the raw row metadata of the requested song
        input_idx = df_filtered[df_filtered['track_name'].str.lower() == song_title.lower()].index[0]
        input_song = df_filtered.iloc[input_idx]
    except IndexError:
        return f"Error: '{song_title}' wasn't found in this dataset. Try another track!"
    
    # Extract the genre of our input song
    input_genre = str(input_song['track_genre']).lower()
    
    # Grab all mathematical similarity scores for this specific song index
    sim_scores = list(enumerate(similarity_matrix[input_idx]))
    
    # Sort from highest score (1.0) down to lowest score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Create an empty basket to hold our filtered winner indices
    filtered_matches = []
    
    # Loop through the sorted results one by one
    for index, score in sim_scores:
        if index == input_idx:
            continue # Skip matching the song with itself!
            
        # Look up the genre of the current candidate song in the loop
        candidate_genre = str(df_filtered.loc[index, 'track_genre']).lower()
        
        # --- THE CULTURAL FILTER LOGIC ---
        # Case A: If the input song is Indian, the candidate MUST be Indian
        if 'indian' in input_genre and 'indian' in candidate_genre:
            filtered_matches.append(index)
            
        # Case B: If the input song is NOT Indian, the candidate MUST NOT be Indian
        elif 'indian' not in input_genre and 'indian' not in candidate_genre:
            filtered_matches.append(index)
            
        # Stop checking files the moment our basket hits the user's requested limit
        if len(filtered_matches) == num_recommendation:
            break
            
    # Look up the final row metadata and return it cleanly
    return df_filtered.iloc[filtered_matches][['track_name', 'artists', 'track_genre']]

# def recommend_song(song_title,num_recommendation=5):

#     try:
#         idx = df_filtered[df_filtered['track_name'].str.lower() == song_title.lower()].index[0]
#     except IndexError:
#         return f"{song_title} wasn't found in this subset. Try another Song !"
    
#     sim_scores =list(enumerate(similarity_matrix[idx]))
#     sim_scores = sorted(sim_scores,key=lambda x : x[1],reverse=True)

#     top_matches = sim_scores[1:num_recommendation+1] 
#     rec_indices = [i[0] for i in top_matches] 
#     return df_filtered.iloc[rec_indices][['track_name', 'artists', 'track_genre']]
recommend_song_with_filter("Raabta")

,track_name,artists,track_genre
5078,Dil Mere,The Local Train,indian
5958,Rahguzar,Adarsh Rao,indian
5232,Khayaal,Arijit Anand;Ankita Barwad,indian
5048,Deva Deva (Malayalam),Pritam;Hesham Abdul Wahab;Arijit Singh;Arya Dh...,indian
5956,Xullo Son,Debabrata Gogoi;Rajnish Saikia,indian
